# Text Preprocessing for Clinical Notes

## Beginner-Friendly Step-by-Step Notebook

This notebook shows how to preprocess clinical notes for healthcare NLP.

We will move through this workflow:

```text
raw clinical note
        ↓
clean text
        ↓
tokenize text
        ↓
remove stopwords + lemmatize
        ↓
extract medical entities
        ↓
create a final summary dataset
```

The goal is **not to memorize code**.  
The goal is to understand the repeatable **code patterns** used in healthcare text preprocessing.

## 1. Install required libraries

This cell installs the libraries used in the notebook.

### What this code does

| Library | Why we use it |
|---|---|
| `pandas` | Stores clinical notes in a table |
| `nltk` | Tokenizes text, removes stopwords, and lemmatizes words |
| `spacy` | Processes text using NLP models |
| `scispacy` | Adds scientific and biomedical NLP support |

### Code pattern to remember

```python
!pip install package_name
```

Use this pattern when a notebook needs a library that is not already installed.

In [ ]:
!pip install -q pandas nltk regex spacy==3.7.2 scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

## Important note about runtime restart

If you are using Google Colab, after installing `scispacy` and the model, you may need to restart the runtime.

In Colab:

```text
Runtime → Restart runtime
```

Then continue running the notebook starting from the next cell.

## 2. Import libraries and download NLTK resources

Before using a tool in Python, we import it.

### What this code does

| Code | Meaning |
|---|---|
| `import pandas as pd` | Use pandas with the nickname `pd` |
| `import re` | Use regular expressions for text cleaning |
| `word_tokenize` | Split text into words |
| `sent_tokenize` | Split text into sentences |
| `stopwords` | Remove common words such as “the”, “is”, and “and” |
| `WordNetLemmatizer` | Convert words to base form |
| `spacy.load(...)` | Load a biomedical NLP model |

### Code pattern to remember

```python
import library
from library.module import specific_tool
```

Pattern:

```text
import tools → download resources → load NLP model
```

In [ ]:
import pandas as pd
import re
import nltk
import warnings
import spacy

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter

warnings.filterwarnings("ignore")

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt_tab")

nlp = spacy.load("en_core_sci_sm")

print("All libraries imported successfully!")
print("Ready to begin clinical text preprocessing analysis.")

## 3. Create a small clinical notes dataset

In real projects, clinical notes may come from an EHR system, CSV file, database, or secure clinical dataset.

Here, we create a small practice dataset manually.

### What this code does

Each dictionary is one row of data.  
The DataFrame is like an Excel table.

| Column | Meaning |
|---|---|
| `note_id` | Unique note number |
| `patient_id` | Patient identifier |
| `note_type` | Type of note |
| `note_text` | The actual clinical note |
| `note_date` | Date of the note |
| `provider_specialty` | Medical specialty |

### Code pattern to remember

```python
data = [
    {"column1": value, "column2": value},
    {"column1": value, "column2": value}
]

df = pd.DataFrame(data)
```

Pattern:

```text
create rows → convert to DataFrame → inspect the data
```

In [ ]:
clinical_notes = [
    {
        "note_id": 1,
        "patient_id": "P001",
        "note_type": "Progress Note",
        "note_text": """Patient presents with c/o severe headache and nausea x 3 days.
        PMH: HTN, DM Type II.
        Meds: Metformin 500mg BID, Lisinopril 10mg QD.
        PE: BP 145/92, HR 88, Temp 98.6F. Patient appears uncomfortable.
        Assessment: Migraine headache, uncontrolled hypertension.
        Plan: Start Sumatriptan 50mg PRN, increase Lisinopril to 20mg QD.""",
        "note_date": "2024-01-15",
        "provider_specialty": "Internal Medicine"
    },
    {
        "note_id": 2,
        "patient_id": "P002",
        "note_type": "Discharge Summary",
        "note_text": """DISCHARGE SUMMARY: 65 y/o M admitted for COPD exacerbation.
        Patient with h/o smoking (40 pack-years), presented with SOB and productive cough.
        Hospital course: Treated with nebulizers, prednisone 40mg daily, and antibiotics (Azithromycin).
        Condition improved significantly.
        Discharge meds: Albuterol inhaler, Advair 250/50 BID, Prednisone taper.
        F/U with pulmonology in 2 weeks.""",
        "note_date": "2024-01-20",
        "provider_specialty": "Pulmonology"
    },
    {
        "note_id": 3,
        "patient_id": "P003",
        "note_type": "History and Physical",
        "note_text": """CC: Chest pain. HPI: 58 y/o F w/ sudden onset chest pain radiating to L arm, assoc. w/ diaphoresis.
        PMH: Hyperlipidemia, Family h/o CAD.
        Exam: Patient diaphoretic, anxious. CV: Regular rate and rhythm, no murmurs.
        EKG: ST elevation in leads II, III, aVF.
        Impression: Acute inferior STEMI.
        Plan: Activate cath lab, start heparin drip, aspirin 325mg, plavix load 600mg.""",
        "note_date": "2024-01-22",
        "provider_specialty": "Cardiology"
    }
]

df = pd.DataFrame(clinical_notes)

print("Clinical Notes Dataset:")
print(f"Total number of notes: {len(df)}")
print(f"Dataset columns: {list(df.columns)}")

print("
First clinical note:")
print(df.iloc[0]["note_text"])

## 4. Display the first rows

### What this code does

`df.head()` shows the first 5 rows of a DataFrame.

### Why we use it

Before analyzing data, always check what the data looks like.

### Code pattern to remember

```python
df.head()
```

Pattern:

```text
after creating or loading a DataFrame → use df.head()
```

In [ ]:
df.head()

## 5. Clean and normalize clinical text

Raw clinical notes can contain capital letters, extra spaces, symbols, and inconsistent formatting.

This function makes the text more consistent.

### What this function does

Step by step:

1. Converts text to lowercase.
2. Replaces extra spaces and line breaks with one space.
3. Removes unwanted symbols but keeps useful medical characters.

We keep characters such as `/` and `-` because clinical notes often contain terms like:

```text
y/o
h/o
pack-years
250/50
```

### Code pattern to remember

```python
def function_name(input):
    clean input
    return cleaned_input

new_column = old_column.apply(function_name)
```

Pattern:

```text
define cleaning function → apply it to a column → save result in new column
```

In [ ]:
def clean_clinical_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^a-z0-9\s.,/\-]", "", text)
    return text


df["cleaned_text"] = df["note_text"].apply(clean_clinical_text)

print("Original text:")
print(df.iloc[0]["note_text"][:300])

print("
Cleaned text:")
print(df.iloc[0]["cleaned_text"][:300])

print("
Text cleaning completed successfully!")

## 6. Tokenize text

Tokenization means splitting text into smaller pieces.

There are two common types:

| Tokenization type | Meaning |
|---|---|
| Sentence tokenization | Split note into sentences |
| Word tokenization | Split note into words |

Example:

```text
patient has headache. patient has nausea.
```

Sentence tokens:

```python
["patient has headache.", "patient has nausea."]
```

Word tokens:

```python
["patient", "has", "headache", ".", "patient", "has", "nausea", "."]
```

### Code pattern to remember

```python
def tokenize_text(text):
    sentences = sent_tokenize(text)
    words = word_tokenize(text)

    return {
        "sentences": sentences,
        "words": words
    }
```

Pattern:

```text
input text → split into sentences → split into words → return both
```

In [ ]:
def tokenize_text(text):
    sentences = sent_tokenize(text)
    words = word_tokenize(text)

    return {
        "sentences": sentences,
        "words": words
    }


sample_tokens = tokenize_text(df.iloc[0]["cleaned_text"])

print("Sentence Tokenization:")
print(f"Number of sentences: {len(sample_tokens['sentences'])}")
print("
First sentence:")
print(sample_tokens["sentences"][0])

print("
" + "=" * 50)

print("
Word Tokenization:")
print(f"Total number of words: {len(sample_tokens['words'])}")
print("
First 15 words:")
print(sample_tokens["words"][:15])

## 7. Remove stopwords and lemmatize tokens

This step makes the word list cleaner and more useful.

It removes:

```text
common words
punctuation
numbers
very short words
```

It also lemmatizes words.

### What are stopwords?

Stopwords are common words that often do not add much meaning.

Examples:

```text
the
is
and
with
for
```

### What is lemmatization?

Lemmatization changes words to their base form.

Examples:

```text
patients → patient
medications → medication
```

### Important note

This beginner version removes numbers such as `500mg` and `145/92`.

In real clinical NLP projects, medication doses and vital signs can be important, so you may choose to keep them.

### Code pattern to remember

```python
new_list = []

for item in old_list:
    if condition_to_skip:
        continue

    changed_item = change(item)
    new_list.append(changed_item)

return new_list
```

Pattern:

```text
start empty list → loop → skip unwanted items → transform useful items → save them
```

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))


def preprocess_tokens(tokens):
    processed_tokens = []

    for token in tokens:
        token = token.lower()

        if token in stop_words:
            continue

        if not token.isalpha():
            continue

        lemma = lemmatizer.lemmatize(token)

        if len(lemma) > 2:
            processed_tokens.append(lemma)

    return processed_tokens


processed_words = preprocess_tokens(sample_tokens["words"])

print("Original token count:", len(sample_tokens["words"]))
print("After stopword removal and lemmatization:", len(processed_words))
print("Reduction:", len(sample_tokens["words"]) - len(processed_words), "tokens removed")

print("
Processed tokens first 20:")
print(processed_words[:20])

## 8. Apply token preprocessing to every note

Now we apply the token preprocessing function to the full DataFrame.

### What this code does

For every cleaned note:

```text
take cleaned text
→ tokenize into words
→ preprocess the words
→ save result in df["tokens"]
```

### Code pattern to remember

```python
df["new_column"] = df["old_column"].apply(
    lambda text: function(text)
)
```

Pattern:

```text
use apply() when every row needs the same transformation
```

In [ ]:
df["tokens"] = df["cleaned_text"].apply(
    lambda text: preprocess_tokens(word_tokenize(text))
)

print("Preprocessing applied to all clinical notes in dataset.")
print(df[["note_id", "tokens"]])

## 9. Extract medical entities using NER

NER means **Named Entity Recognition**.

It finds important terms in text, such as diseases, symptoms, procedures, and medications.

Examples of possible medical entities:

```text
headache
nausea
Metformin
Lisinopril
COPD
chest pain
STEMI
```

### Why we use the original note text

For entity extraction, we use the original text instead of the cleaned text because capitalization and punctuation may help NLP models understand meaning.

### Code pattern to remember

```python
def extract_entities(text):
    doc = model(text)

    results = []

    for item in doc.ents:
        results.append({
            "text": item.text,
            "label": item.label_
        })

    return results
```

Pattern:

```text
send text to NLP model → loop through found entities → store useful details
```

In [ ]:
def extract_medical_entities(text):
    doc = nlp(text)

    entities = []

    for entity in doc.ents:
        entities.append({
            "text": entity.text,
            "label": entity.label_,
            "start": entity.start_char,
            "end": entity.end_char
        })

    return entities


df["entities"] = df["note_text"].apply(extract_medical_entities)

print("Named Entities Extracted from First Clinical Note:")
print("=" * 60)

for entity in df.iloc[0]["entities"]:
    print(f"Entity: {entity['text']:<30} | Type: {entity['label']}")

print("
" + "=" * 60)
print("Entity Extraction Summary:")

for index, row in df.iterrows():
    print(f"Note {row['note_id']}: {len(row['entities'])} entities found")

## 10. Create a preprocessing summary table

After preprocessing, we create a summary to check what happened.

### What this summary includes

| Column | Meaning |
|---|---|
| `note_id` | Note number |
| `note_type` | Type of clinical note |
| `original_length` | Number of characters before cleaning |
| `cleaned_length` | Number of characters after cleaning |
| `token_count` | Number of processed tokens |
| `entity_count` | Number of medical entities found |

### Code pattern to remember

```python
summary_df = pd.DataFrame({
    "column_name": df["source_column"],
    "new_metric": df["source_column"].apply(function)
})
```

Pattern:

```text
choose useful columns → calculate metrics → build summary table
```

In [ ]:
summary_df = pd.DataFrame({
    "note_id": df["note_id"],
    "note_type": df["note_type"],
    "original_length": df["note_text"].apply(len),
    "cleaned_length": df["cleaned_text"].apply(len),
    "token_count": df["tokens"].apply(len),
    "entity_count": df["entities"].apply(len)
})

print("Preprocessing Summary:")
print(summary_df.to_string(index=False))

print("
" + "=" * 60)
print("Overall Statistics:")
print(f"Total notes processed: {len(df)}")
print(f"Average tokens per note: {summary_df['token_count'].mean():.1f}")
print(f"Average entities per note: {summary_df['entity_count'].mean():.1f}")
print(f"Total entities extracted: {summary_df['entity_count'].sum()}")

print("
Preprocessed dataset is ready for analysis!")

## 11. Exercise 1: Preprocess a new clinical note

This exercise applies the same cleaning and preprocessing steps to a new diabetes follow-up note.

### What this code does

```text
raw note
→ clean note
→ tokenize note
→ remove stopwords and lemmatize
```

### Code pattern to remember

```python
cleaned = clean_function(raw_text)
tokens = word_tokenize(cleaned)
processed = preprocess_tokens(tokens)
```

Pattern:

```text
clean → tokenize → preprocess
```

In [ ]:
new_note = """Patient is a 45 y/o M with Type 2 Diabetes Mellitus presenting for routine follow-up.
HbA1c today is 7.8%, up from 7.2% three months ago. Patient reports poor dietary compliance.
Current medications: Metformin 1000mg BID, Glipizide 5mg daily.
Plan: Refer to diabetes educator, increase Metformin to 1000mg TID, recheck HbA1c in 3 months."""

cleaned_new_note = clean_clinical_text(new_note)

new_note_words = word_tokenize(cleaned_new_note)

processed_new_note_tokens = preprocess_tokens(new_note_words)

print("Cleaned note:")
print(cleaned_new_note)

print("
Processed tokens:")
print(processed_new_note_tokens)

print("
Number of processed tokens:")
print(len(processed_new_note_tokens))

## 12. Exercise 2: Group entities by entity type

This exercise organizes extracted entities into groups.

### Why we use it

Grouping entities makes results easier to read and analyze.

Instead of one long list, we organize terms by label.

### Code pattern to remember

```python
grouped = {}

for item in items:
    group_name = item["label"]

    if group_name not in grouped:
        grouped[group_name] = []

    grouped[group_name].append(item)
```

Pattern:

```text
create empty dictionary → loop → create group if missing → add item to group
```

In [ ]:
second_note_entities = df.iloc[1]["entities"]

entities_by_type = {}

for entity in second_note_entities:
    label = entity["label"]
    text = entity["text"]

    if label not in entities_by_type:
        entities_by_type[label] = []

    entities_by_type[label].append(text)

print("Entities grouped by type:")

for label, entity_texts in entities_by_type.items():
    print(f"
{label}:")
    for text in entity_texts:
        print(f" - {text}")

## 13. Exercise 3: Compare token frequencies

This exercise finds the most common processed tokens across all notes.

### Why we use it

Token frequency helps us understand what words appear most often.

### Important beginner idea

`extend()` adds each item from one list into another list.

Example:

```python
all_tokens.extend(["pain", "nausea"])
```

This adds:

```text
pain
nausea
```

### Code pattern to remember

```python
from collections import Counter

all_items = []

for item_list in df["column"]:
    all_items.extend(item_list)

counts = Counter(all_items)
top_items = counts.most_common(10)
```

Pattern:

```text
combine all lists → count items → get most common
```

In [ ]:
all_tokens = []

for token_list in df["tokens"]:
    all_tokens.extend(token_list)

token_frequencies = Counter(all_tokens)

top_10_tokens = token_frequencies.most_common(10)

print("Top 10 Most Frequent Tokens Across All Notes:")
print("=" * 60)

for token, frequency in top_10_tokens:
    print(f"{token:<20} {frequency:>10}")

print(f"
Total unique tokens: {len(token_frequencies)}")
print(f"Total tokens processed: {len(all_tokens)}")

## 14. Exercise 4: Build a full preprocessing pipeline

A pipeline combines several steps into one reusable function.

Instead of writing this every time:

```python
cleaned = clean_clinical_text(text)
tokens = word_tokenize(cleaned)
processed = preprocess_tokens(tokens)
entities = extract_medical_entities(text)
```

we can just write:

```python
results = full_preprocessing_pipeline(text)
```

### Code pattern to remember

```python
def pipeline(raw_input):
    step1_result = step1(raw_input)
    step2_result = step2(step1_result)
    step3_result = step3(step2_result)

    return {
        "step1": step1_result,
        "step2": step2_result,
        "step3": step3_result
    }
```

Pattern:

```text
input → step 1 → step 2 → step 3 → return organized results
```

In [ ]:
def full_preprocessing_pipeline(raw_text):
    cleaned_text = clean_clinical_text(raw_text)

    word_tokens = word_tokenize(cleaned_text)

    processed_tokens = preprocess_tokens(word_tokens)

    entities = extract_medical_entities(raw_text)

    results = {
        "cleaned_text": cleaned_text,
        "tokens": processed_tokens,
        "token_count": len(processed_tokens),
        "entities": entities,
        "entity_count": len(entities)
    }

    return results

## 15. Test the full pipeline

After writing a function, always test it.

This confirms the pipeline works from beginning to end.

### Code pattern to remember

```python
result = function(input)

print(result["key1"])
print(result["key2"])
```

Pattern:

```text
call function → save result → inspect each output
```

In [ ]:
test_note = """Patient presents with acute dyspnea and chest pain.
History of CAD and recent MI. Started on aspirin and statin therapy."""

results = full_preprocessing_pipeline(test_note)

print("Preprocessing Pipeline Results:")
print("=" * 60)

print("Cleaned text:")
print(results["cleaned_text"])

print("
Token count:")
print(results["token_count"])

print("
Tokens:")
print(results["tokens"])

print("
Entity count:")
print(results["entity_count"])

print("
Entities extracted:")
for entity in results["entities"]:
    print(f" - {entity['text']} ({entity['label']})")

## Final review: Main code patterns from this notebook

### Pattern 1: Install and import

```python
!pip install package
import package
```

Use when you need an external library.

---

### Pattern 2: Create a table

```python
data = [
    {"column": value},
    {"column": value}
]

df = pd.DataFrame(data)
```

Use when you want structured data.

---

### Pattern 3: Write a reusable function

```python
def function_name(input):
    result = do_something(input)
    return result
```

Use when you repeat the same logic many times.

---

### Pattern 4: Apply a function to a DataFrame column

```python
df["new_column"] = df["old_column"].apply(function_name)
```

Use when every row needs the same transformation.

---

### Pattern 5: Loop through a list and keep useful items

```python
new_list = []

for item in old_list:
    if bad_condition:
        continue

    new_list.append(item)
```

Use when filtering tokens, rows, or entities.

---

### Pattern 6: Store multiple outputs in a dictionary

```python
return {
    "output_1": value_1,
    "output_2": value_2
}
```

Use when a function gives back several results.

---

### Pattern 7: Build a pipeline

```python
def pipeline(raw_input):
    step1 = function1(raw_input)
    step2 = function2(step1)
    step3 = function3(step2)

    return step3
```

Use when one task has several steps.

---

## Most important idea

Do not memorize the code.

Remember the workflow:

```text
1. Load text
2. Clean text
3. Split text into tokens
4. Remove noise
5. Normalize words
6. Extract medical entities
7. Summarize results
```

That pattern appears in many healthcare NLP projects.